In [2]:
# Parse packet loss
path_to_log = r"C:\\Users\\Administrator\\Downloads\\sfu_scal_logs\\20251104T103745Z\\provider_sfu1\\sfu_mdc_log_20251104_113542.log"
start_offset=10
duration = 100 
from pathlib import Path
from collections import defaultdict
import shlex
from datetime import datetime, timezone

def parse_line_to_dict(line):
    """Parse a single log line into a dict of key->value.
    Values may be quoted; we use shlex.split to respect quotes.
    Only parts containing '=' are kept."""
    parts = shlex.split(line)
    d = {}
    for p in parts:
        if '=' in p:
            k, v = p.split('=', 1)
            d[k] = v
    return d

def group_records_by_id(path):
    """Read the file at `path` and group parsed records by the 'id' field.
    Returns a defaultdict(list) mapping id -> list of record dicts (without the 'id' key).
    If a line has no id, it is grouped under '__no_id__'.
    Lines starting with '#' or empty lines are skipped.
    """
    p = Path(path)
    grouped = defaultdict(list)
    if not p.exists():
        # File not found: print a message and return empty grouping
        print(f"Log file not found: {p}")
        return grouped

    with p.open('r', encoding='utf-8', errors='replace') as f:
        for line in f:
            line = line.strip()
            if not line or line.startswith('#'):
                continue
            d = parse_line_to_dict(line)
            if 'id' in d:
                rec_id = d.pop('id')
            else:
                rec_id = '__no_id__'
            grouped[rec_id].append(d)
    return grouped

# Run the grouping and expose `grouped_records` in the notebook namespace





In [3]:
def filter_records_by_time(records, start_ts, duration):
    """Filter records to those within start_ts and start_ts + duration"""
    end_ts = start_ts + duration
    filtered = []
    for record in records:
        if 'ts' in record:
            ts = int(record['ts'])
            if start_ts <= ts <= end_ts:
                filtered.append(record)
    return filtered

def filter_records_by_status(records, status_code):
    """Filter records to those with a specific status code."""
    filtered = []
    for record in records:
        if 'status' in record:
            try:
                status = int(record['status'])
                if status == status_code:
                    filtered.append(record)
            except ValueError:
                pass
    return filtered

def extract_field_as_array(records, field):
    """Extract a field from each record as a list of floats.
    Records missing the field are skipped."""
    arr = []
    for record in records:
        if field in record:
            try:
                val = float(record[field])
                arr.append(val)
            except ValueError:
                pass
    return arr

In [4]:

# Load SFU and client data into dicts
from pathlib import Path
import numpy as np

base_path = "C:\\Users\\Administrator\\Downloads\\sfu_scal_logs\\mbix2"
records_all = []
for d in Path(base_path).iterdir():
    if not d.is_dir():
        continue
    sfu_records = {}
    client_records = {}
    for dd in d.iterdir():
        if dd.is_dir() and "provider" in dd.name.lower():
            files = [p for p in dd.iterdir() if p.is_file()]
            log_file = files[0]
            grouped_records = group_records_by_id(log_file)
            key_record = filter_records_by_status(grouped_records["SessionManagerConnection"], 0)
            key = key_record[0]["providerKey"]
            start_ts = int(grouped_records["Init"][0]["ts"])
            sfu_object = {
                "start_ts": start_ts,
                "records": grouped_records
            }
            sfu_records[key] = sfu_object
        if dd.is_dir() and "client" in dd.name.lower():
            files = [p for p in dd.iterdir() if p.is_file()]
            for log_file in files:
                grouped_records = group_records_by_id(log_file)
                key_record = filter_records_by_status(grouped_records["SessionManagerConnection"], 0)
                key = key_record[0]["preferredclientID"]
                client_object = {
                    "start_ts": int(grouped_records["Init"][0]["ts"]),
                    "records": grouped_records
                }
                client_records[key] = client_object

    records_all.append({
        "sfu": sfu_records,
        "client": client_records
    })

In [5]:
# Process all SFU SystemResources records
memUsage_all = []
cpuUsage_all = []
for record_set in records_all:
    sfu_records = record_set["sfu"].values()
    for sfu in sfu_records:
        start_ts = sfu["start_ts"]
        recs = filter_records_by_time(sfu["records"]["SystemResources"], start_ts+100*1000, 120*1000)
        memUsage = extract_field_as_array(recs, "memUsage")
        cpuUsage = extract_field_as_array(recs, "cpuUsage")
        memUsage_all.extend(memUsage)
        cpuUsage_all.extend(cpuUsage)
        print("=================")
        print(f"Processed {log_file}:\nmemUsage mean={np.mean(memUsage):.2f}, std={np.std(memUsage):.2f}; cpuUsage mean={np.mean(cpuUsage):.2f}, std={np.std(cpuUsage):.2f}")
            
print("====================================")
print(f"Overall memUsage: mean={np.mean(memUsage_all):.2f}, std={np.std(memUsage_all):.2f}")
print(f"Overall cpuUsage: mean={np.mean(cpuUsage_all):.2f}, std={np.std(cpuUsage_all):.2f}")

Processed C:\Users\Administrator\Downloads\sfu_scal_logs\mbix2\20251109T155446Z\provider_sfu1\sfu_mdc_log_20251109_165242.log:
memUsage mean=8598.00, std=10.31; cpuUsage mean=65.00, std=0.00
Overall memUsage: mean=8598.00, std=10.31
Overall cpuUsage: mean=65.00, std=0.00


In [19]:
# Client SFU Stats
packet_loss_all = []
n_clients_all = []
n_clients_bin = {}
clients = {}
for record_set in records_all:
    sfu_records = record_set["sfu"].values()
    for sfu in sfu_records:
        packet_loss = []
        n_clients = []
        start_ts = sfu["start_ts"]
        recs = filter_records_by_time(sfu["records"]["SFU"], start_ts+100*1000, 120*1000)
        recs = filter_records_by_status(recs, 9001) 
        for r in recs:
            stats = r["stats"]
            s = stats.strip()
            if s.startswith('[') and s.endswith(']'):
                s = s[1:-1].strip()
            else:
                continue
            client_segs = s.split(';')
            for s in client_segs:
                if s == "":
                    continue
                fields = s.split('@')
                client_dict = {}
                for field in fields:
                    if '=' in field:
                        k, v = field.split('=', 1)
                        client_dict[k.strip()] = v.strip()
                client_id = int(client_dict["client"])
                if client_id not in clients:
                    clients[client_id] = {
                        "active_tracks": [],
                        "loss": []
                    }
                
                client_obj = clients[client_id]
                client_obj["loss"].append(float(client_dict["avgLoss"]))
                #print(client_id, float(client_dict["bitrate"])/1000000)
                active_tracks = client_dict["activeTracks"][1:-3].strip()
                client_obj["active_tracks"].append(active_tracks.split('|'))
for client_id in sorted(clients):
    client_val = clients[client_id]
    packet_loss_all.extend(client_val["loss"])
    for a in client_val["active_tracks"]:
        n_clients_all.append(len(a))
        n_clients_bin[len(a)] = n_clients_bin.get(len(a), 0) + 1
    #print(f"Client {client_id}: max loss={np.max(client_val['loss']):.2f}, mean loss={np.mean(client_val['loss']):.2f}, std={np.std(client_val['loss']):.2f}")
print("====================================")
print(f"Overall packet loss: mean={np.mean(packet_loss_all):.2f}, std={np.std(packet_loss_all):.2f}")
print(f"Overall number of active tracks: mean={np.mean(n_clients_all):.2f}, std={np.std(n_clients_all):.2f}")
print(f"Active tracks distribution: {n_clients_bin}")

Overall packet loss: mean=0.06, std=0.21
Overall number of active tracks: mean=9.31, std=4.91
Active tracks distribution: {1: 89, 2: 3, 13: 21, 14: 30, 15: 20, 16: 32, 17: 20, 18: 7, 10: 64, 11: 56, 9: 107, 7: 61, 5: 31, 6: 49, 8: 73, 4: 5, 12: 40, 19: 6, 20: 8, 21: 4, 29: 1, 31: 1, 26: 1, 22: 2, 23: 3, 24: 2}


In [7]:
# End-to-end latency
latency_all = []
for record_set in records_all:
    client_records = record_set["client"]
    sent_tracks = {}
    received_tracks = {}
    for client_key in client_records:
        client = client_records[client_key]
        start_ts = client["start_ts"]
        frames_by_time = filter_records_by_time(client["records"]["SFUConnection"], start_ts+120*1000, 120*1000)
        fully_received_frames = filter_records_by_status(frames_by_time, 6003)
        sent_frames = filter_records_by_status(frames_by_time, 6000)
        for r in sent_frames:
            track_id = r["trackID"]
            frame_nr = r["frame"]
            ts_sent = int(r["ts"])
            if track_id not in sent_tracks:
                sent_tracks[track_id] = {}
            sent_tracks[track_id][frame_nr] = ts_sent
        for r in fully_received_frames:
            track_id = r["trackID"]
            if track_id == "":
                continue
            frame_nr = r["frame"]
            ts_received = int(r["ts"])
            if track_id not in received_tracks:
                received_tracks[track_id] = {}
            if frame_nr not in received_tracks[track_id]:
                received_tracks[track_id][frame_nr] = []
            received_tracks[track_id][frame_nr].append({"client": client_key, "ts": ts_received})
    for track in sent_tracks:
        for frame_nr in sent_tracks[track]:
            ts_sent = sent_tracks[track][frame_nr]
            if track in received_tracks and frame_nr in received_tracks[track]:
                ts_received_list = received_tracks[track][frame_nr]
                for ts_received in ts_received_list:
                    latency = ts_received["ts"] - ts_sent
                    latency_all.append(latency)
                    #if ts_received['client'] == "12" and track == "cl2_mdc_video_0_0":
                    #if latency < 20:
                    #    print(f"Track {track} Frame {frame_nr}: Latency = {latency} ms {ts_received['client']} {ts_sent} {ts_received['ts']}")
print("====================================")
print(f"Overall latency: mean={np.mean(latency_all):.2f} ms, std={np.std(latency_all):.2f} ms")

Overall latency: mean=6.69 ms, std=2.52 ms
